<a id="introduction"></a>

# PS4: Restricted Boltzmann Machines and MNIST Digit Recall

Handwritten digit images from the [MNIST database](https://en.wikipedia.org/wiki/MNIST_database) contain rich spatial structure — stroke patterns, curves, and intersections that co-occur across examples of the same digit. In this problem set, we explore how two associative memory models — the classical Hopfield network and the restricted Boltzmann machine (RBM) — store and recall digit patterns from corrupted inputs.

The [Hopfield network](https://en.wikipedia.org/wiki/Hopfield_network) stores patterns in a Hebbian weight matrix and retrieves them by minimizing an energy function through iterative state updates. The [restricted Boltzmann machine](https://en.wikipedia.org/wiki/Restricted_Boltzmann_machine) takes a different approach: it learns a probability distribution over binary pixel patterns and reconstructs inputs via block Gibbs sampling. We compare both models on the same recall task — recovering a corrupted MNIST digit — and then examine how model capacity and training data affect reconstruction quality.

> __Learning Objectives__
> 
> By the end of this problem set, you should be able to:
>
> * __Hopfield network recall:__ Build a memory matrix from MNIST digit patterns and apply Hopfield network dynamics to reconstruct stored patterns from corrupted inputs. Analyze how the number of stored patterns affects recall quality.
> * __RBM training with contrastive divergence:__ Train a small restricted Boltzmann machine on MNIST digit images using the CD-1 algorithm and evaluate reconstruction accuracy on corrupted test patterns. Compare performance against the classical Hopfield network.
> * __Pretrained RBM recall:__ Load a pretrained RBM and use block Gibbs sampling to reconstruct corrupted MNIST digits. Compare recall quality against the small trained RBM across different corruption fractions.

Let's get started!
___

<a id="setup"></a>

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and sets the random seed. For more information, see the [VLDataScienceMachineLearningPackage.jl documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/).

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) for RBM operations and MNIST data loading. Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material. Because multiple packages export a `sample` function, we use the explicit module prefix `VLDataScienceMachineLearningPackage.sample(...)` for RBM Gibbs sampling.

### Constants
We define constants that control the experiment setup, model architecture, and training configuration. The comment next to each constant describes its purpose, units, and permissible values.

In [ ]:
# Image dimensions (MNIST: 28 x 28 pixels)
number_of_rows = 28; # number of rows in the image
number_of_cols = 28; # number of columns in the image
number_of_pixels = number_of_rows * number_of_cols; # total pixels per image (784)

# Data loading
number_of_examples = 50; # patterns to store in memory matrix (Task 1)
digit_for_experiment = 3; # MNIST digit to use (0-9)
digit_to_compare = 8; # second digit for cross-model comparison
number_of_training_examples = 50; # training examples for small RBM (Task 2)

# Small RBM architecture (Task 2)
number_of_visible = number_of_pixels; # 784 visible units
number_of_hidden_small = 128; # hidden units in small RBM
number_of_epochs = 20; # training epochs
batchsize_train = 10; # mini-batch size
number_of_updates_per_epoch = ceil(Int, number_of_training_examples / batchsize_train);
η_train = 0.01; # learning rate
β_train = 1.0; # inverse temperature during training

# Recall parameters
number_of_gibbs_recall = 10; # Gibbs steps for RBM recall (enough to denoise, avoids mode-mixing)
β_recall = 1.0; # inverse temperature for recall
corruption_frac = 0.30; # default corruption fraction
number_of_hopfield_steps = 20; # Hopfield dynamics steps

# Visualization
number_of_show = 6; # number of stored patterns to visualize

### Implementation
The notebook uses local helper functions for MNIST image handling:

> * `decode(s; number_of_rows, number_of_columns)`: Converts a flattened state vector back to a 2D image matrix for visualization.
> * `corrupt(v, frac; seed)`: Flips a random fraction of bits in a binary pattern to simulate noisy input.
> * `tanimoto(a, b)`: Computes the Tanimoto (Jaccard) similarity between two binary vectors, measuring recall quality.

> __`decode(s; number_of_rows, number_of_columns)`__
>
> Converts a flattened state vector $\mathbf{s}$ (with values in $\{-1, +1\}$ or $\{0, 1\}$) back to a 2D image matrix. The function reshapes the vector, transposes to match row-major ordering, and replaces $-1$ values with $0$ for grayscale display.

Let's implement the `decode(...)` function:

In [ ]:
function decode(s::Vector{<:Number}; number_of_rows::Int=28, number_of_columns::Int=28)::Array{<:Number,2}
    
    # initialize -
    X = reshape(s, number_of_rows, number_of_columns) |> X -> transpose(X) |> Matrix;
    X̂ = replace(X, -1 => 0);

    # return -
    return X̂
end;

> __`corrupt(v, frac; seed)`__
>
> Corrupts a binary $\{-1, +1\}$ pattern $\mathbf{v}$ by flipping a random fraction `frac` of its bits. Each flipped bit changes sign: $v_i \leftarrow -v_i$. This simulates noisy or partially observed inputs for the recall task.

Let's implement the `corrupt(...)` function:

In [ ]:
function corrupt(v::Vector{Int64}, frac::Float64; seed::Int=42)::Vector{Int64}
    
    # initialize -
    n = length(v);
    number_of_flips = round(Int, frac * n);
    rng = MersenneTwister(seed);
    flip_idx = StatsBase.sample(rng, 1:n, number_of_flips, replace=false);
    
    # corrupt the pattern -
    v_corrupted = copy(v);
    v_corrupted[flip_idx] .*= -1;

    # return -
    return v_corrupted
end;

> __`tanimoto(a, b)`__
>
> Computes the Tanimoto (Jaccard) similarity between two $\{0, 1\}$ binary vectors. The Tanimoto coefficient is $T(\mathbf{a}, \mathbf{b}) = \frac{\langle \mathbf{a}, \mathbf{b} \rangle}{\|\mathbf{a}\|_1 + \|\mathbf{b}\|_1 - \langle \mathbf{a}, \mathbf{b} \rangle}$, which equals 0 when the patterns share no "on" pixels and 1 when they are identical. Unlike bit accuracy, Tanimoto is not inflated by background pixels.

Let's implement the `tanimoto(...)` function:

In [ ]:
function tanimoto(a::Vector{<:Number}, b::Vector{<:Number})::Float64
    
    # initialize -
    ab = dot(a, b);
    denom = sum(a) + sum(b) - ab;

    # return -
    return denom == 0 ? 0.0 : ab / denom
end;

___
<a id="task1"></a>

## Task 1: Memory Matrix and Hopfield Recall
A [Hopfield network](https://en.wikipedia.org/wiki/Hopfield_network) stores binary patterns as a symmetric weight matrix and retrieves them by iteratively updating each unit's state to minimize a global energy function. Given a set of $N$ stored patterns $\left\{\boldsymbol{\xi}^{(1)},\ldots,\boldsymbol{\xi}^{(N)}\right\}$ with $\boldsymbol{\xi}^{(\mu)}\in\{-1,+1\}^{n}$, the Hebbian learning rule constructs the weight matrix as an outer-product sum:

$$
\mathbf{W} = \frac{1}{N}\sum_{\mu=1}^{N}\boldsymbol{\xi}^{(\mu)}\left(\boldsymbol{\xi}^{(\mu)}\right)^{\top},\qquad w_{ii}=0.
$$

Recall dynamics update the state as $\mathbf{v}^{(t+1)} = \text{sign}\!\left(\mathbf{W}\,\mathbf{v}^{(t)}\right)$, which is guaranteed to converge to a local minimum of the energy function $E(\mathbf{v}) = -\frac{1}{2}\mathbf{v}^{\top}\mathbf{W}\,\mathbf{v}$. In this task, we build a memory matrix from MNIST digit examples and test whether Hopfield dynamics can recover a stored pattern from a corrupted input.

> __Memory matrix construction__
>
> We load `number_of_examples` images of `digit_for_experiment` from the MNIST dataset, threshold each pixel at $0.5$ to obtain a binary $\{0, 1\}$ image, convert to $\{-1, +1\}$ encoding, and store each flattened pattern as a column of $\mathbf{X}_{\pm 1} \in \mathbb{Z}^{n_{pixels} \times N}$.

Let's load the MNIST data and build the memory matrix:

In [ ]:
digits_dict, X, X_pm1 = let

    # initialize -
    digits_dict = MyMNISTHandwrittenDigitImageDataset(number_of_examples = number_of_examples + 10);
    X = zeros(Float64, number_of_pixels, number_of_examples); # {0,1} encoding
    X_pm1 = zeros(Int64, number_of_pixels, number_of_examples); # {-1,+1} encoding

    # TODO: fill in X and X_pm1
    # for each i in 1:number_of_examples
    #   get the i-th image of digit_for_experiment from digits_dict
    #   flatten to a vector, threshold at 0.5 to get a binary {0,1} pattern
    #   convert to {-1,+1} and store in X_pm1[:, i]

    println("Memory matrix: $(size(X_pm1, 1)) pixels x $(size(X_pm1, 2)) stored patterns")

    # return -
    (digits_dict, X, X_pm1)
end;

> __Visualize stored patterns__
>
> Before running recall, we display a subset of the stored patterns to confirm the memory matrix was constructed correctly. Each column of $\mathbf{X}\in\{0,1\}^{n_{\text{pixels}}\times N}$ is reshaped to a $28\times 28$ image using the `decode(...)` function.

Let's visualize the first `number_of_show::Int` stored patterns.

In [ ]:
let
    # initialize -
    p_list = [];

    # build the heatmaps -
    for k in 1:number_of_show
        img = decode(X[:, k]);
        p = heatmap(img, c=:grays, colorbar=false, aspect_ratio=1,
                    title="Pattern $k", axis=false, ticks=false);
        push!(p_list, p);
    end

    # return -
    plot(p_list..., layout=(1, number_of_show), size=(660, 130))
end

> __Hopfield weight matrix (Hebbian rule)__
>
> The Hopfield weight matrix is $\mathbf{W} = \frac{1}{N} \mathbf{X}_{\pm 1} \mathbf{X}_{\pm 1}^{\top} \in \mathbb{R}^{n_{pixels} \times n_{pixels}}$, where $N$ is the number of stored patterns. The diagonal is zeroed to prevent trivial self-reinforcement. Recall dynamics update the state as $\mathbf{v}^{(t+1)} = \text{sign}(\mathbf{W} \mathbf{v}^{(t)})$ until convergence or for `number_of_hopfield_steps` iterations.

Let's compute the Hopfield weight matrix:

In [ ]:
W_hopfield = let

    # initialize -
    N = size(X_pm1, 2); # number of stored patterns

    # TODO: compute the Hebbian weight matrix W = X_pm1 * X_pm1' / N
    # then zero the diagonal
    W = zeros(number_of_pixels, number_of_pixels); # replace this

    # return -
    W
end;

> __Recall from a corrupted input__
>
> We select a held-out test image (not used to build the memory matrix), corrupt it by flipping `corruption_frac` $= 30\%$ of its bits using the `corrupt(...)` function, and run Hopfield dynamics for `number_of_hopfield_steps::Int` $= 20$ iterations. The Tanimoto similarity between the recalled pattern and the original measures recall quality on a scale from $0$ (no overlap) to $1$ (identical).

Let's run the Hopfield recall experiment.

In [ ]:
v_original, v_corrupted, v_recalled = let

    # initialize - grab a held-out image (index number_of_examples + 1)
    img = digits_dict[digit_for_experiment][:, :, number_of_examples + 1];
    v = Float64.(img)[:];
    b = (v .> 0.5);
    v_original = Int64.(2 .* b .- 1);

    # corrupt the pattern -
    v_corrupted = corrupt(v_original, corruption_frac; seed=1);

    # TODO: run Hopfield dynamics for number_of_hopfield_steps steps
    # v_state <- sign(W_hopfield * v_state) at each step
    # break ties by setting 0 values to 1
    v_state = copy(v_corrupted); # placeholder - replace with dynamics

    println("Corruption fraction           : $(corruption_frac)")
    println("Tanimoto similarity (Hopfield recall): $(round(tanimoto(Float64.(((v_state) .+ 1) ./ 2), Float64.(((v_original) .+ 1) ./ 2)), digits=3))")

    # return -
    (v_original, v_corrupted, v_state)
end;

In [ ]:
let
    # initialize -
    imgs = [decode(v_original), decode(v_corrupted), decode(v_recalled)];
    titles = ["Original",
              "Δ ($(round(Int, corruption_frac*100))%)",
              "Hopfield Recall"];

    # build the heatmaps -
    p_list = [heatmap(img, c=:grays, colorbar=false, aspect_ratio=1,
                      title=t, titlefontsize=9, axis=false, ticks=false)
              for (img, t) in zip(imgs, titles)];

    # return -
    plot(p_list..., layout=(1, 3), size=(540, 180))
end

___
<a id="task2"></a>

## Task 2: Train a Small RBM
The Hopfield network encodes patterns through a deterministic outer-product rule and retrieves them by energy minimization. This approach is limited by a fixed storage capacity and produces hard binary outputs with no measure of uncertainty. A [restricted Boltzmann machine (RBM)](https://en.wikipedia.org/wiki/Restricted_Boltzmann_machine) takes a different approach: it learns a probability distribution over binary patterns from training data and reconstructs inputs via block Gibbs sampling. The bipartite structure of the RBM (no within-layer connections) enables efficient parallel updates of all hidden or all visible units in a single step.

We train a small RBM with `number_of_visible::Int` $= 784$ visible units and `number_of_hidden_small::Int` $= 128$ hidden units on `number_of_training_examples::Int` $= 50$ images of digit `digit_for_experiment::Int` using contrastive divergence (CD-1). After training, we reconstruct the same corrupted held-out digit from Task 1 and compare recall quality.

> __Training data preparation__
>
> We load `number_of_training_examples` images of `digit_for_experiment` from the MNIST dataset and convert them to a $\{-1, +1\}$ integer matrix of shape $(n_{pixels} \times n_{train})$. A uniform `Categorical` distribution over the training indices serves as the data sampling distribution for the `learn(...)` function.

Let's prepare the training data:

In [ ]:
X_train_pm1, p_train = let

    # initialize -
    X_train_pm1 = zeros(Int64, number_of_pixels, number_of_training_examples);

    # TODO: fill in X_train_pm1
    # for each i in 1:number_of_training_examples
    #   get the i-th image of digit_for_experiment, threshold at 0.5, convert to {-1,+1}

    p_train = Categorical(number_of_training_examples);

    println("Training data: $(size(X_train_pm1, 1)) pixels x $(size(X_train_pm1, 2)) examples")

    # return -
    (X_train_pm1, p_train)
end;

> __Contrastive divergence training (CD-1)__
>
> We initialize the RBM with small random weights and zero biases, then train for `number_of_epochs` epochs. Each epoch runs `number_of_updates_per_epoch` mini-batch updates using the `learn(...)` function with `T=2` Gibbs steps (CD-1, since the package counts the initial state as step 1). After each epoch, we compute the mean reconstruction error (fraction of bits incorrectly reconstructed) on a 5-image probe set.

Let's train the small RBM:

In [ ]:
rbm_small, reconstruction_errors = let

    # initialize - build W (diag = 0)
    W_init = 0.01 * randn(number_of_visible, number_of_hidden_small);
    b_init = zeros(number_of_hidden_small);
    a_init = zeros(number_of_visible);

    # build the model -
    rbm = build(MyRestrictedBoltzmannMachineModel, (
        W = W_init, # weights
        b = b_init, # hidden biases
        a = a_init  # visible biases
    ));

    # setup for reconstruction error tracking -
    number_of_probe = min(5, number_of_training_examples);
    probe_idx = 1:number_of_probe;
    reconstruction_errors = Float64[];

    # training loop -
    for epoch in 1:number_of_epochs

        # TODO: call learn(...) to update rbm for one epoch
        # use: maxnumberofiterations=number_of_updates_per_epoch, T=2, β=β_train,
        #      batchsize=batchsize_train, η=η_train, tol=1e-10, verbose=false

        # TODO: compute reconstruction error on probe_idx
        # for each probe, call VLDataScienceMachineLearningPackage.sample(rbm, v0; T=2, β=β_train)
        # compare V_rec[:, end] with v0 and compute the mean bit error
        push!(reconstruction_errors, 0.0); # replace with computed error

        epoch % 5 == 0 && println("Epoch $(lpad(epoch,3)) | recon error = $(round(reconstruction_errors[end], digits=4))");
    end

    # return -
    (rbm, reconstruction_errors)
end;

> __Training convergence__
>
> The reconstruction error (fraction of incorrectly reconstructed bits on a probe set) should decrease or plateau over training epochs as the RBM learns the co-occurrence structure of the digit patterns.

Let's plot the reconstruction error over epochs.

In [ ]:
let
    plot(1:number_of_epochs, reconstruction_errors,
         xlabel = "Training Epoch",
         ylabel = "Reconstruction Error (bit fraction)",
         title  = "Small RBM Training Convergence ($(number_of_visible) -> $(number_of_hidden_small))",
         label  = "Recon. Error",
         lw = 2, color = :blue, marker = :circle, size = (650, 380))
end

> __Small RBM recall__
>
> We apply the same corruption used in Task 1 to the held-out digit and reconstruct it using `number_of_gibbs_recall::Int` $= 50$ steps of block Gibbs sampling with the trained small RBM. This tests whether the RBM learned a distribution that can denoise corrupted inputs.

Let's run the small RBM recall experiment.

In [ ]:
v_rbm_recalled = let

    # initialize - same corruption as Task 1
    v0_pm1 = corrupt(v_original, corruption_frac; seed=1);

    # TODO: call VLDataScienceMachineLearningPackage.sample(rbm_small, v0_pm1; T=number_of_gibbs_recall, β=β_recall)
    # extract V_rec[:, end] as the reconstructed visible state
    v_rec = copy(v0_pm1); # placeholder - replace

    println("Bit accuracy (small RBM vs original): $(round(mean(v_rec .== v_original), digits=3))")

    # return -
    v_rec
end;

In [ ]:
let
    # initialize -
    imgs = [decode(v_original), decode(v_corrupted), decode(v_rbm_recalled)];
    titles = ["Original",
              "Δ ($(round(Int, corruption_frac*100))%)",
              "Small RBM"];

    # build the heatmaps -
    p_list = [heatmap(img, c=:grays, colorbar=false, aspect_ratio=1,
                      title=t, titlefontsize=9, axis=false, ticks=false)
              for (img, t) in zip(imgs, titles)];

    # return -
    plot(p_list..., layout=(1, 3), size=(540, 180))
end

___
<a id="task3"></a>

## Task 3: Recall with a Pretrained RBM
The small RBM from Task 2 was trained on only `number_of_training_examples` $= 50$ examples of a single digit class. A larger RBM with more hidden units and broader training data should learn richer co-occurrence patterns and produce better recall across a wider range of inputs. We load a pretrained RBM ($784$ visible $\rightarrow$ $512$ hidden units) that was trained on all ten MNIST digit classes using CD-1 for $100$ epochs. The pretrained model is stored in `data/pretrained_rbm_mnist.jld2`.

> __Note__: If the pretrained model file does not exist, run `julia scripts/pretrain_mnist_rbm.jl` from the project root to generate it.

> __Loading the pretrained RBM__
>
> We load the saved weight matrix $\mathbf{W}\in\mathbb{R}^{n\times m}$, visible bias vector $\mathbf{a}\in\mathbb{R}^{n}$, and hidden bias vector $\mathbf{b}\in\mathbb{R}^{m}$ from the JLD2 file and reconstruct a `MyRestrictedBoltzmannMachineModel` instance using the `build(...)` function.

Let's load the pretrained RBM and store it in the `rbm_pretrained::MyRestrictedBoltzmannMachineModel` variable.

In [ ]:
rbm_pretrained = let
    
    # initialize -
    rbm_file = jldopen(joinpath(_PATH_TO_DATA, "pretrained_rbm_mnist.jld2")); # load the pretrained RBM file
    
    # build the RBM model using the loaded parameters -
    W_loaded = rbm_file["W"]; # weights
    b_loaded = rbm_file["b"]; # hidden biases
    a_loaded = rbm_file["a"]; # visible biases
    close(rbm_file);

    rbm = build(MyRestrictedBoltzmannMachineModel, (
        W = W_loaded, # weights
        b = b_loaded, # hidden biases
        a = a_loaded  # visible biases
    ));

    println("Loaded pretrained RBM: $(number_of_visible) -> $(size(rbm.W, 2)) hidden units")

    # return -
    rbm
end;

> __Pretrained RBM recall__
>
> We apply the same held-out corrupted digit to the pretrained RBM and reconstruct it using `number_of_gibbs_recall::Int` $= 50$ steps of block Gibbs sampling. We compare the Tanimoto similarity against both the Hopfield network (Task 1) and the small RBM (Task 2).

Let's run the pretrained RBM recall experiment.

In [ ]:
v_pretrained_recalled = let

    # initialize - same corruption as Tasks 1 and 2
    v0_pm1 = corrupt(v_original, corruption_frac; seed=1);

    # TODO: call VLDataScienceMachineLearningPackage.sample(rbm_pretrained, v0_pm1; T=number_of_gibbs_recall, β=β_recall)
    # extract V_rec[:, end]
    v_rec = copy(v0_pm1); # placeholder - replace

    println("Bit accuracy (pretrained RBM vs original): $(round(mean(v_rec .== v_original), digits=3))")

    # return -
    v_rec
end;

In [ ]:
let
    # initialize -
    imgs = [decode(v_original), decode(v_corrupted), decode(v_pretrained_recalled)];
    titles = ["Original",
              "Δ ($(round(Int, corruption_frac*100))%)",
              "Pre-RBM"];

    # build the heatmaps -
    p_list = [heatmap(img, c=:grays, colorbar=false, aspect_ratio=1,
                      title=t, titlefontsize=9, axis=false, ticks=false)
              for (img, t) in zip(imgs, titles)];

    # return -
    plot(p_list..., layout=(1, 3), size=(540, 180))
end

> __Tanimoto similarity vs. corruption fraction__
>
> We sweep corruption fractions from $10\%$ to $80\%$ and run `number_of_trials::Int` $= 10$ recall trials at each level. For each trial, we corrupt a held-out digit, reconstruct it with each model, and compute the Tanimoto similarity against the original image. The sweep reveals how each model's recall quality degrades as the input becomes increasingly corrupted.

Let's compute the sweep and store the results in the `accuracy_pretrained::Vector{Float64}` and `accuracy_small::Vector{Float64}` variables.

In [ ]:
accuracy_pretrained, accuracy_small, cf_range = let

    # initialize -
    cf_range = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80];
    number_of_trials = 10;
    number_of_total = size(digits_dict[digit_for_experiment], 3);
    test_start = number_of_examples + 2;
    accuracy_pretrained = Float64[];
    accuracy_small = Float64[];

    # sweep over corruption fractions -
    for cf in cf_range
        total_pretrained = 0.0;
        total_small = 0.0;

        for trial in 1:number_of_trials

            # initialize - grab a test image and corrupt it
            idx = min(test_start + trial - 1, number_of_total);
            img = digits_dict[digit_for_experiment][:, :, idx];
            v = Float64.(img)[:];
            b = (v .> 0.5);
            v_original = Int64.(2 .* b .- 1);
            v_corrupted = corrupt(v_original, cf; seed=trial);

            # TODO: reconstruct v_corrupted with rbm_pretrained (T=number_of_gibbs_recall, β=β_recall)
            # add tanimoto similarity to total_pretrained

            # TODO: reconstruct v_corrupted with rbm_small (T=number_of_gibbs_recall, β=β_recall)
            # add tanimoto similarity to total_small
        end

        push!(accuracy_pretrained, total_pretrained / number_of_trials);
        push!(accuracy_small, total_small / number_of_trials);
    end

    # return -
    (accuracy_pretrained, accuracy_small, cf_range)
end;

In [ ]:
let
    # plot accuracy vs corruption fraction for both models
    plot(cf_range, accuracy_pretrained,
        xlabel  = "Corruption Fraction",
        ylabel  = "Tanimoto Similarity",
        title   = "Recall Accuracy vs. Corruption Fraction (digit $(digit_for_experiment))",
        label   = "Pretrained RBM (784 -> 512)",
        lw=2, marker=:circle, color=:steelblue, size=(650, 380));
    
    plot!(cf_range, accuracy_small,
         label   = "Small RBM (784 -> $(number_of_hidden_small))",
         lw=2, marker=:square, color=:orange, linestyle=:dash);
end

> __Cross-digit comparison__
>
> The small RBM was trained only on digit `digit_for_experiment::Int`. The pretrained RBM was trained on all ten digit classes. We repeat the accuracy sweep on `digit_to_compare::Int` — a digit the small RBM has never seen. The pretrained RBM should recall this unseen digit well; the small RBM should not, because its learned distribution does not cover that digit class.

Let's compute the cross-digit sweep.

In [ ]:
accuracy_pretrained_d2, accuracy_small_d2, _ = let

    # initialize -
    cf_range_local = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80];
    number_of_trials = 10;
    number_of_total = size(digits_dict[digit_to_compare], 3);
    accuracy_pretrained = Float64[];
    accuracy_small = Float64[];

    # sweep over corruption fractions -
    for cf in cf_range_local
        total_pretrained = 0.0;
        total_small = 0.0;

        for trial in 1:number_of_trials

            # initialize - grab a test image and corrupt it
            idx = min(trial, number_of_total);
            img = digits_dict[digit_to_compare][:, :, idx];
            v = Float64.(img)[:];
            b = (v .> 0.5);
            v_original = Int64.(2 .* b .- 1);
            v_corrupted = corrupt(v_original, cf; seed=trial);

            # TODO: reconstruct v_corrupted with rbm_pretrained (T=number_of_gibbs_recall, β=β_recall)
            # add tanimoto similarity to total_pretrained

            # TODO: reconstruct v_corrupted with rbm_small (T=number_of_gibbs_recall, β=β_recall)
            # add tanimoto similarity to total_small
        end

        push!(accuracy_pretrained, total_pretrained / number_of_trials);
        push!(accuracy_small, total_small / number_of_trials);
    end

    # return -
    (accuracy_pretrained, accuracy_small, cf_range_local)
end;

In [ ]:
let
    # initialize -
    p1 = plot(cf_range, accuracy_pretrained,
              label="Pretrained RBM", lw=2, marker=:circle, color=:steelblue,
              title="Digit $(digit_for_experiment) (training digit)",
              xlabel="Corruption Fraction", ylabel="Tanimoto Similarity",
              ylims=(0, 1.05));
    plot!(p1, cf_range, accuracy_small,
          label="Small RBM", lw=2, marker=:square, color=:orange, linestyle=:dash);

    p2 = plot(cf_range, accuracy_pretrained_d2,
              label="Pretrained RBM", lw=2, marker=:circle, color=:steelblue,
              title="Digit $(digit_to_compare) (unseen digit)",
              xlabel="Corruption Fraction", ylabel="Tanimoto Similarity",
              ylims=(0, 1.05));
    plot!(p2, cf_range, accuracy_small_d2,
          label="Small RBM", lw=2, marker=:square, color=:orange, linestyle=:dash);

    # return -
    plot(p1, p2, layout=(1, 2), size=(900, 380))
end

___
<a id="discussion"></a>

## Discussion
Use the results from Tasks 1-3 to answer the discussion questions below.

**DQ1: What is the storage capacity limit of the Hopfield network?** The classical Hopfield network has a storage capacity of approximately $0.138 \times n_{\text{pixels}}$ random binary patterns before recall quality degrades significantly. For MNIST images ($n_{\text{pixels}} = 784$), this limit is approximately $108$ patterns.

> __Strategy__: Modify `number_of_examples` in the constants block to values above and below $108$. Re-run Task 1 and observe how Tanimoto similarity changes. Describe the failure mode that occurs when capacity is exceeded.

In [ ]:
# DQ1 Answer:
# TODO: answer DQ1 here

In [ ]:
did_I_answer_DQ1 = false;  # TODO: update to true if answered DQ1 {true | false}

**DQ2: What do the RBM hidden units detect?** Each column $\mathbf{W}[:,k]\in\mathbb{R}^{784}$ of the weight matrix of `rbm_small` encodes what pixel pattern activates hidden unit $k$. Reshape each column to $28 \times 28$ and visualize it as an image.

> __Strategy__: Use `heatmap` to display several columns of `rbm_small.W` reshaped to $28 \times 28$. Look at the high-magnitude positive and negative weights. Describe whether the units appear to detect localized stroke features, global templates, or something else.

In [ ]:
# DQ2 Answer:
# TODO: answer DQ2 here

In [ ]:
did_I_answer_DQ2 = false;  # TODO: update to true if answered DQ2 {true | false}

**DQ3: Specialization vs. generalization.** The two-panel plot compares Tanimoto similarity for `digit_for_experiment` (the small RBM's training digit) and `digit_to_compare` (a digit the small RBM never saw).

> __Strategy__: At $30\%$ corruption, compare the four similarity values across the two panels. Explain why the small RBM wins (or ties) on its training digit but fails on the unseen digit. Explain why the pretrained RBM performs more consistently across both digits. What does this reveal about the tradeoff between specialization and generalization in associative memory?

In [ ]:
# DQ3 Answer:
# TODO: answer DQ3 here

In [ ]:
did_I_answer_DQ3 = false;  # TODO: update to true if answered DQ3 {true | false}

___
<a id="summary"></a>

## Summary
This problem set applied the classical Hopfield network and restricted Boltzmann machines to MNIST digit recall, comparing how deterministic energy minimization and probabilistic generative modeling handle pattern reconstruction from corrupted inputs.

> __Key Takeaways__
>
> * **Hopfield network capacity:** The Hopfield network stores patterns in a Hebbian weight matrix and retrieves them via energy minimization. Storage capacity is bounded by approximately $0.138 \times n_{\text{pixels}}$ patterns, which constrains reliable recall to a small number of stored digits relative to the full MNIST library.
> * **RBM as a generative memory:** An RBM learns a probability distribution over binary pixel patterns and reconstructs inputs via block Gibbs sampling. Contrastive divergence training allows the RBM to generalize beyond exact stored patterns to nearby corrupted inputs.
> * **Scale improves recall:** A pretrained RBM with more hidden units and training data maintains higher Tanimoto similarity across a wider range of corruption fractions than a small locally-trained model. Increased capacity enables the model to capture more complex co-occurrence patterns in the digit images.

The specialization-generalization tradeoff observed in Task 3 motivates architectures that balance model capacity with training data breadth for robust pattern recall.

___
<a id="tests"></a>

## Tests
Run this cell after completing all tasks to verify your results.

In [ ]:
let
    using Test
    @testset verbose = true "CHEME 5820 PS4 test suite" begin

        @testset "Setup and Data" begin
            @test typeof(digits_dict) <: Dict
            @test size(X_pm1) == (number_of_pixels, number_of_examples)
            @test all(x -> x == 1 || x == -1, X_pm1)
        end

        @testset "Task 1: Hopfield Network" begin
            @test size(W_hopfield) == (number_of_pixels, number_of_pixels)
            @test issymmetric(W_hopfield)
            @test all(W_hopfield[diagind(W_hopfield)] .== 0.0)
            @test length(v_original) == number_of_pixels
            @test all(x -> x == 1 || x == -1, v_recalled)
        end

        @testset "Task 2: Small RBM" begin
            @test size(rbm_small.W) == (number_of_visible, number_of_hidden_small)
            @test length(reconstruction_errors) == number_of_epochs
            @test all(0 .<= reconstruction_errors .<= 1.0)
            @test length(v_rbm_recalled) == number_of_pixels
        end

        @testset "Task 3: Pretrained RBM" begin
            @test size(rbm_pretrained.W, 1) == number_of_visible
            @test length(v_pretrained_recalled) == number_of_pixels
            @test length(accuracy_pretrained) == length(cf_range)
            @test length(accuracy_small) == length(cf_range)
        end

        @testset "Discussion" begin
            @test did_I_answer_DQ1 == true
            @test did_I_answer_DQ2 == true
            @test did_I_answer_DQ3 == true
        end
    end
end